# 응답 객체 상세 확인

`LLMResponse`, `to_dict()`, `finish_reason`, `function_calls` 등 응답 구조 탐색.

In [ ]:
from llm_manager._internal import handler_client as llm_handler
from llm_manager._internal.handler_client import call

# llm_handler.configure(base_url="http://localhost:8100")


In [19]:
response = call("안녕하세요", system="간결하게 답변하세요.", thinking="minimal")

print(f"text: {response.text}")
print(f"model: {response.model}")
print(f"finish_reason: {response.finish_reason}")
print(f"request_id: {response.request_id}")
print(f"usage: {response.usage}")
if response.usage:
    print(f"  input_tokens: {response.usage.input_tokens}")
    print(f"  output_tokens: {response.usage.output_tokens}")
    print(f"  total_tokens: {response.usage.total_tokens}")
    print(f"  thinking_tokens: {response.usage.thinking_tokens}")

text: 안녕하세요! 무엇을 도와드릴까요?
model: gemini-3.5-flash
finish_reason: STOP
request_id: e0146dce-3735-4e8c-857a-7f80698fcb56
usage: input_tokens=9 output_tokens=8 total_tokens=17 thinking_tokens=None
  input_tokens: 9
  output_tokens: 8
  total_tokens: 17
  thinking_tokens: None


In [16]:
response = call("$Z[\sqrt{-5}]$ 가 UFD가 아님을 설명해줘", system="자세하게 답변하세요.", thinking="high")

print(f"text: {response.text}")
print(f"model: {response.model}")
print(f"finish_reason: {response.finish_reason}")
print(f"request_id: {response.request_id}")
print(f"usage: {response.usage}")
if response.usage:
    print(f"  input_tokens: {response.usage.input_tokens}")
    print(f"  output_tokens: {response.usage.output_tokens}")
    print(f"  total_tokens: {response.usage.total_tokens}")
    print(f"  thinking_tokens: {response.usage.thinking_tokens}")

text: 정수론에서 $\mathbb{Z}[\sqrt{-5}]$가 **유일 인수 분해 정역(Unique Factorization Domain, UFD)**이 아님을 증명하는 것은 대수적 수론의 매우 고전적이고 중요한 예시입니다.

이를 상세히 설명하기 위해 먼저 필요한 도구들을 정의하고, 구체적인 반례를 통해 증명하겠습니다.

---

### 1. 기본 정의 및 도구

#### (1) 노름 (Norm) 함수
$\mathbb{Z}[\sqrt{-5}]$의 임의의 원소 $\alpha = a + b\sqrt{-5}$ ($a, b \in \mathbb{Z}$)에 대하여, 노름 함수 $N$을 다음과 같이 정의합니다.
$$N(a + b\sqrt{-5}) = (a + b\sqrt{-5})(a - b\sqrt{-5}) = a^2 + 5b^2$$
노름 함수는 다음과 같은 중요한 성질을 가집니다.
*   $N(\alpha\beta) = N(\alpha)N(\beta)$ (곱셈적 성질)
*   $N(\alpha) = 1$일 때만 $\alpha$는 가역원(unit)이다.
*   $\mathbb{Z}[\sqrt{-5}]$에서 $a^2 + 5b^2 = 1$을 만족하는 정수해는 $a=\pm 1, b=0$뿐이므로, **가역원은 $\{1, -1\}$뿐**입니다.

#### (2) 기약 원소 (Irreducible Element)
어떤 원소가 가역원이 아니면서, 두 가역원이 아닌 원소의 곱으로 나타낼 수 없을 때 이를 기약 원소라고 합니다.

---

### 2. 유일 인수 분해의 실패 (구체적 예시)

숫자 **6**을 두 가지 다른 방법으로 인수분해해 보겠습니다.

1.  $6 = 2 \times 3$
2.  $6 = (1 + \sqrt{-5})(1 - \sqrt{-5})$

만약 $\mathbb{Z}[\sqrt{-5}]$가 UFD라면, 이 두 인수분해에 등장하는 인수들은 순서를 바꾸거나 가역원($\pm 1$)을 곱한 차이만 있어야 합니다. 하지만 실제로는 그렇지 않음을 보이겠습니다.

-

### JSON 응답의 `to_dict()`

In [ ]:
response = call(
    "서울의 위도와 경도를 알려줘",
    schema={"type": "object", "properties": {"lat": {"type": "number"}, "lng": {"type": "number"}}},
)
data = response.to_dict()
print(f"to_dict(): {data}")
print(f"type: {type(data)}")

### Gemini 3 신규 응답 필드 — `tool_calls`, `grounding`, `code_blocks`, `assistant_message`

Gemini 3 server-side 도구를 쓰면 응답에 추가 필드가 채워진다.
각 필드 의미와 등장 시점:

| 필드 | 의미 | 등장 |
| --- | --- | --- |
| `tool_calls` | server-side tool 호출 흔적 (`tool_type` + `args` + `id`) | mixed paradigm |
| `tool_responses` | tool_call.id 짝 (`response` dict) | mixed paradigm |
| `grounding` | google_search 인용 (queries + chunks + supports) | google_search 사용 |
| `code_blocks` | code_execution 코드 (`language` + `code`) | code_execution 사용 |
| `code_results` | code 실행 결과 (`outcome` + `output`) | code_execution 사용 |
| `assistant_message` | role + parts 통째 echo (round-trip 용) | 모든 응답 |


In [ ]:
# google_search 시 grounding 필드 채워짐
response = call("오늘 환율 알려줘", tools=[{"google_search": {}}])

print("text:", response.text[:200], "...")
print()
print("grounding:")
if response.grounding:
    print(f"  쿼리: {response.grounding.web_search_queries}")
    print(f"  chunks: {len(response.grounding.chunks)} 개")
    for c in response.grounding.chunks[:3]:
        print(f"    - {c.title}: {c.uri[:60]}...")
    print(f"  supports: {len(response.grounding.supports)} 개 (citation span)")


In [ ]:
# code_execution 시 code_blocks + code_results 채워짐
response = call("100 까지의 소수의 개수를 계산해줘", tools=[{"code_execution": {}}])

print("text:", response.text[:300], "...")
print()
print("code_blocks:")
if response.code_blocks:
    for blk in response.code_blocks:
        print(f"  [{blk.language}]")
        print("  " + blk.code.replace("\n", "\n  "))
print("code_results:")
if response.code_results:
    for r in response.code_results:
        print(f"  outcome={r.outcome}, output={r.output[:100]!r}")


### `assistant_message` — round-trip 용 통째 echo

`response.assistant_message` 는 응답을 `Content(role="model", content=list[Part])` 로 묶은 것.
Turn 2 history 에 `r.assistant_message.model_dump(exclude_none=True)` 한 줄로 넣으면
role + 모든 part type + thought_signature 자동 보존.
**`Session.send_tool_results()` 가 내부적으로 이걸 활용**.


In [ ]:
tools = [{
    "name": "get_stock_price",
    "description": "주가 조회",
    "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]},
}]

response = call("삼성전자(005930) 주가 알려줘", tools=tools)

print("assistant_message:")
if response.assistant_message:
    print(f"  role: {response.assistant_message.role}")
    for i, p in enumerate(response.assistant_message.content):
        print(f"  parts[{i}]: type={p.type}")
        if p.type == "function_call":
            print(f"      name={p.function_call.name}, args={p.function_call.args}")
            ts = p.function_call.thought_signature
            print(f"      thought_signature (base64): {ts[:30] + '...' if ts else None}")
